# Tutorial 3. Custom Components (`JSComponent` and `ReactComponent`)

::::{note}
:icon: false

#### Build a Motion-powered component in Panel

Built-in widgets are great until you need a custom interaction pattern.  
In this tutorial, you will wrap an external animation library (**Motion**) inside Panel and keep it synchronized with Python state.

You will:

- build a custom animated "Like" component with `JSComponent`
- send structured events from browser -> Python
- rebuild the same idea with `ReactComponent`
::::

## Learning goals

After this tutorial, you should be able to:

1. Wrap a third-party frontend library inside Panel.
2. Decide when to use `JSComponent` vs `ReactComponent`.
3. Sync component state and events between Python and the browser.
4. Compose custom components with regular Panel objects.
5. Use a dev workflow that supports fast iteration.

---

In [ ]:
import panel as pn
import param

pn.extension()

## 1) Mental model: custom component = Python API + frontend renderer

Both `JSComponent` and `ReactComponent` use the same architecture:

1. Define a Python class with Parameters (`param.Integer`, `param.String`, ...)
2. Provide frontend rendering code in `_esm`
3. Let Panel synchronize state between Python and the browser

```{tip}
Use `JSComponent` for lightweight DOM logic. Use `ReactComponent` when JSX and state hooks make the UI easier to maintain.
```

## 2) A cool example: wrap Motion in `JSComponent`

This component uses [Motion](https://motion.dev/) to animate a button every time it is clicked, while keeping the like count in a synced Python Parameter.

In [ ]:
from panel.custom import JSComponent


class MotionLikeButton(JSComponent):
    likes = param.Integer(default=0)
    last_clicked_iso = param.String(default="")

    _stylesheets = ["""
    button {
        font-size: 1.1rem;
        border: none;
        border-radius: 999px;
        padding: 10px 16px;
        background: #ec4899;
        color: white;
        cursor: pointer;
        box-shadow: 0 6px 18px rgba(236, 72, 153, 0.35);
    }
    """]

    _esm = """
    import { animate } from "https://esm.sh/motion@11";

    export function render({ model }) {
      const btn = document.createElement("button");

      const sync = () => {
        btn.textContent = `❤️  ${model.likes} likes`;
      };

      btn.addEventListener("click", () => {
        model.likes += 1;
        model.send_msg({ iso: new Date().toISOString(), likes: model.likes });

        animate(btn, { scale: [1, 1.14, 1], rotate: [0, -4, 4, 0] }, {
          duration: 0.42,
          easing: "ease-out"
        });
      });

      model.on("likes", sync);
      sync();
      return btn;
    }
    """

    def _handle_msg(self, msg):
        self.last_clicked_iso = msg["iso"]

like = MotionLikeButton()
pn.Column(
    like,
    pn.pane.Str(like.param.likes),
    pn.pane.Markdown(like.param.last_clicked_iso),
)

Why this example matters:

- external JS library (`motion`) works directly in `_esm`
- animation runs in browser for smooth UX
- business state (`likes`) stays in Python

## 3) Browser -> Python events and messages

In custom components, you have two common options:

- `model.send_event("<name>", event)` for DOM-style events
- `model.send_msg(data)` for custom payloads (often simpler)

The Motion example used `send_msg` to push a timestamp and updated count to Python.

```{note}
Use `_handle_<event_name>` for `send_event`, and `_handle_msg` for `send_msg`.
```


## 4) Same idea with `ReactComponent`

`ReactComponent` extends `JSComponent` and adds JSX plus `model.useState("<parameter>")`.

In [ ]:
from panel.custom import ReactComponent


class ReactLikeButton(ReactComponent):
    likes = param.Integer(default=0)

    _esm = """
    export function render({ model }) {
      const [likes, setLikes] = model.useState("likes");
      return (
        <button
          style={{
            border: "none",
            borderRadius: "999px",
            padding: "10px 16px",
            background: "#6366f1",
            color: "white",
            cursor: "pointer"
          }}
          onClick={() => setLikes(likes + 1)}
        >
          🚀 {likes} boosts
        </button>
      );
    }
    """

ReactLikeButton()

## 5) Compose custom components with Panel content

Custom components are just Panel components, so they compose naturally with panes, widgets, and layouts.

In [ ]:
pn.Row(
    pn.Column("### Campaign A", MotionLikeButton()),
    pn.Column("### Campaign B", ReactLikeButton()),
)

This is how you incrementally add custom UX without rewriting your app.

## 6) Fast iteration workflow (`_esm` files + `--dev`)

For non-trivial components, keep code split across files:

- `my_component.py` for Python Parameters and handlers
- `my_component.js` or `my_component.jsx` for frontend logic
- optional `my_component.css` in `_stylesheets`

Serve with:

```bash
panel serve my_component.py --dev
```

`--dev` auto-reloads on changes, which is essential for frontend iteration speed.

## 7) Quick decision guide

- Choose `JSComponent` when:
  - you want direct DOM control
  - logic is compact
  - React tooling is unnecessary
- Choose `ReactComponent` when:
  - JSX improves readability
  - UI has multiple interactive subparts
  - hooks-style state management is a better fit

## Mini challenge

Build an animated "Vote Chip" with an external library:

1. Use `JSComponent` + Motion to animate on click.
2. Track both `votes` and `last_voted_at` in Python.
3. Add a second version in `ReactComponent`.
4. Place both inside one `pn.Row` and compare ergonomics.

## Recap

You now have a practical pattern for advanced UI in Panel:

- wrap external frontend libraries from `_esm`
- keep authoritative state in Python Parameters
- choose `JSComponent` or `ReactComponent` based on complexity
- compose custom pieces with regular Panel building blocks

This is the bridge between Python-first apps and modern frontend interactions.